# Tokenizer

In this section, we are going to build a simple tokenizer that allows us to tokenize our sentences into smaller components called tokens and decode input tokens into the original string. Each token has its own unique identifier.

In the tokenizer, we have two main processes:
- **Encoding**: Chunking our sentences into smaller components called token. Each token will be represented with a unique identifier
<br>
<img height="400px" width="700px" src="images/tokenizer_encoding.png"/>
- **Decoder**: Converting token ids into the original sentence

In this notebook, we will build a tokenizer using an algorithm called **Byte Pair Encoding**, which is used in building many large language models such as ChatGPT or Gemini.


## Importing dataset

Now let's build our tokenizer vocabulary by training it with more text data

In [ ]:
urls = {
    "Education and the good life": "https://www.gutenberg.org/cache/epub/70302/pg70302.txt",
    "The School and Society": "https://www.gutenberg.org/cache/epub/53910/pg53910.txt",
    "What Is and What Might Be": "https://www.gutenberg.org/cache/epub/20555/pg20555.txt",
    "How we think": "https://www.gutenberg.org/cache/epub/37423/pg37423.txt",
    "The Reform of Education": "https://www.gutenberg.org/cache/epub/36762/pg36762.txt"
}

# Build a script that allows you to extract and construct training dataset

In [10]:
# Import book text dataset 
with open("dataset/how_we_think.txt", 'r', encoding='utf-8-sig') as f:
    lines = f.readlines()
    # Merge those lines together
    text = "".join(lines)
    

### Byte Pair Encoding

#### Materials
- [Byte Pair Encoding Hugging Face](https://www.youtube.com/watch?v=HEikzVL-lZU)

#### Core ideas

**Rule 1**: Do not split frequently used words into smaller subwords

**Rule 2**: Split the rare words into smaller, meaningful subwords

- Eg: "play" should not be split. "playing" should be split into "play" and "ing"

#### Benefits
1. "plays" and "playing" comes from the same root "play"
2. Some words have different root words but share the suffix part such as "classification" and "location" which both share "cation"

**BFE algorithm**: Most common pair of consecutive bytes of data is replaced with a byte that does not occur in the data

In [20]:
# Build a Tokenizer Class
class Tokenizer:
    def __init__(self):
        self.token_to_id = {'<start>' : 1, '<end_of_text>' : 2, ' ' : 3, '<unk>' : 4}
        self.id_to_token = {1 : '<start>', 2 : '<end_of_text>', 3 : ' ', 4: "<unk>"}
        self.vocab = set(['<start>', '<end_of_text>', '<unk>', ' '])
        self.bp_merges = {}

    def train(self, text):
        assert len(text) > 0, "You must input a non-empty text"
        text_characters = []
        for char in text:
            if char not in self.vocab:
                self.vocab.add(char)
                id = len(self.vocab)
                self.token_to_id[char] = id
                self.id_to_token[id] = char
            text_characters.append(char)

        while True:
            frequency = {}
            # Contruct frequency from the text_characters
            for i in range(1, len(text_characters)):
                pair = (text_characters[i-1], text_characters[i])
                if pair not in frequency:
                    frequency[pair] = 0
                frequency[pair] += 1
            if not frequency: break

            most_commmon_pair, occurence = max(frequency.items(), key = lambda item: item[1])
            if occurence > 1:
                new_token = ''.join(most_commmon_pair)
                self.vocab.add(new_token)
                id = len(self.vocab)
                self.token_to_id[new_token] = id
                self.id_to_token[id] = new_token
                self.bp_merges[most_commmon_pair] = id # id here is the rank for our pair
                # Merge those tokens inside the text_characters
                new_text_characters = []

                index = 0
                while (index < len(text_characters)):
                    if (text_characters[index] == most_commmon_pair[0] and index < len(text_characters) - 1 and text_characters[index+1] == most_commmon_pair[1]):
                        new_text_characters.append(new_token)
                        index += 2 # Skip the next character
                    else:
                        new_text_characters.append(text_characters[index])
                        index += 1
                text_characters = new_text_characters
            else:
                break

    def encode(self, text: str, add_special_tokens = True):
        assert self.bp_merges, "You must train your tokenizer first!"
        tokens = list(text)
        # Merge based on the rank that a pair was constructed
        while True:
            best_rank = float("inf")
            best_pair = None
            candidate_index = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                rank = self.bp_merges.get(pair, None)
                if rank is not None and rank < best_rank:
                    best_pair = pair
                    best_rank = rank
                    candidate_index = i
            if best_pair is None: break
            # Merge pair with lowest rank
            tokens[candidate_index] = ''.join(best_pair)
            del tokens[candidate_index + 1]

        ids = [self.token_to_id.get(token, self.token_to_id['<unk>']) for token in tokens]
        if add_special_tokens:
            ids = [self.token_to_id["<start>"]] + ids + [self.token_to_id['<end_of_text>']]

        return ids

    def decode(self, inputs):
        string =  "".join([self.id_to_token.get(id, self.id_to_token[4]) for id in inputs])
        return string

In [21]:
tokenizer = Tokenizer()
# Train tokenizer
tokenizer.train(text)

In [26]:
sentence = "The lesson here is under the"
ids = tokenizer.encode(sentence)
[tokenizer.id_to_token[id] for id in ids]

['<start>',
 'The ',
 'lesson ',
 'h',
 'ere is ',
 'under',
 ' the',
 '<end_of_text>']

In [27]:
# ids = [1, 59, 109, 2590, 1275, 66, 257, 25, 2]
tokenizer.decode(ids)

'<start>The lesson here is under the<end_of_text>'


### Positional encoding

- Implement positional encoding: [Link](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)

In [1]:
# Implement Positional Encoding Layer that takes in input and return the output with added position informatio
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def forward(self, X):
        pass

### Create input-target pair for training

In [ ]:
from torch.utils.data import Dataset, DataLoader

class EducationDataset(Dataset):
    def __init__(self):
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, index):
        return super().__getitem__(index)




### MultiHead - Self-attention mechanism

- Implement self-attention with Q, K, V approach

→ Revise on how to implement Masked multi-head attention which prevents the model from attending to subsequent time step from the current time step


In [ ]:
# Set contant 
D_MODEL = 512

In [ ]:
class Self_Attention(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def forward(self, X):
        pass
    
    
    
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads = 8,  *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.attentions = [Self_Attention() for i in range(self.num_heads)]
    def forward(self, X):
        
        pass


In [4]:
class MiniGPT(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def forward(self, X):
        pass


# 3. Training model

**Train your model**

- Pretrained model for a few peochs
- Log training/validation loss and compute **perplexity**.
- Save checkpoints and final model.

**Generate text samples:**

- Use your trained model to generate coherent text related to your chosen domain.
- Show 3–5 examples with different prompts.
- Optionally experiment with **temperature, top-k, top-p sampling**

In [ ]:
EPOCHS = 5
LR = 1e-3
device = 'cpu'
model = MiniGPT()
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
criterion = nn.CrossEntropyLoss()

def train_step(data_loader, model, optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in data_loader:
        inputs = inputs.to(device = device)
        labels = labels.to(device = device)

        raw_logits = model(inputs)
        loss = criterion(raw_logits, labels)

        total_loss += loss.item()
        # Add perplexity metric or BLEU here

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return total_loss / len(data_loader)


def eval_step(data_loader, model):
    model.eval()
    with torch.inference_mode():
        total_loss = 0
        for inputs, labels in data_loader:
            inputs = inputs.to(device=device)
            labels = labels.to(device=device)

            raw_logits = model(inputs)
            loss = criterion(raw_logits, labels)
            total_loss += loss.item()
            
            # Add perplexity metric or BLEU here
        return total_loss / len(data_loader)

In [ ]:
for epoch in range(EPOCHS):
    train_loss = train_step(train_loader, model, optimizer)
    val_loss = eval_step(val_loader, model)
    # Logging model performance 
    print(f"Epoch: [{epoch}|{EPOCHS}]: Train loss = {train_loss} - Validation loss = {val_loss}")
    


## **4. Evaluation:**

- Report quantitative results (loss, perplexity).
- Qualitative evaluation: human judgment of coherence and relevance.